In [0]:
import pandas as pd
from pyspark.sql import functions as F
import re
#%pip install xlrd>=2.0.1
%pip install lxml
%pip install html5lib


In [0]:
def flatten_columns(columns):
    flattened = []

    for column in columns:
        if isinstance(column, tuple):
            parts = [
                str(level).strip()
                for level in column
                if str(level).strip()
                and str(level).strip().lower() != "nan"
                and not str(level).strip().startswith("Unnamed:")
            ]

            name = "_".join(parts)

        else:
            name = str(column).strip()

        # Replace only characters that Delta doesn't allow
        name = re.sub(r"[ ,;{}()\n\t=]", "_", name)

        flattened.append(name)

    return flattened

In [0]:
seasons = ['2122', '2223', '2324', '2425', '2526']
leagues = ['bl', 'll', 'lo', 'pl', 'sa']
stats_types = ['goalkeeping_stats', 'miscellaneous', 'playing_time', 'shooting_stats', 'standard_stats', 'standings']

In [0]:
for season in seasons:
    for league in leagues:
        for stat in stats_types:

            file_path = (
                f"/Volumes/football_data_platform/raw/source_files/"
                f"fbref/{league}_{season}_{stat}.xls"
            )

            tables = pd.read_html(file_path)
            pdf = tables[0]

            # Flatten the complete column hierarchy
            pdf.columns = flatten_columns(pdf.columns)

            df = spark.createDataFrame(pdf)

            bronze_df = (
                df
                .withColumn("_season", F.lit(season))
                .withColumn("_league", F.lit(league))
                .withColumn("_stat", F.lit(stat))
                .withColumn("_ingested_at", F.current_timestamp())
                .withColumn("_source_file", F.lit(file_path))
                .withColumn("_source_system", F.lit("fbref"))
            )

            (
                bronze_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(
                    f"football_data_platform.bronze.fbref_{stat}_{league}"
                )
            )